<a href="https://colab.research.google.com/github/htcmrl/SesTanima/blob/main/SesTanima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q speechbrain torchaudio transformers librosa

import os
import torch
import torchaudio
import librosa
import torch.nn.functional as F
from speechbrain.inference.speaker import EncoderClassifier
from transformers import WhisperProcessor, WhisperForConditionalGeneration

print("SİSTEM BAŞLATILIYOR...")
device = "cuda" if torch.cuda.is_available() else "cpu"

print("1. Modeller RAM'e Yükleniyor (Bu işlem bir kez yapılır)")
kimlik_modeli = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb",
    run_opts={"device": device}
)

whisper_islemci = WhisperProcessor.from_pretrained("openai/whisper-tiny")
whisper_modeli = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
print("Motor Hazır! Sistem giriş isteklerini bekliyor.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 31.6 MB/s eta 0:00:00


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


SİSTEM BAŞLATILIYOR...
1. Modeller RAM'e Yükleniyor (Bu işlem bir kez yapılır)


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Motor Hazır! Sistem giriş isteklerini bekliyor.


In [ ]:
import os
import torch
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier

print("1. Klasörler Hazırlanıyor...")
kaynak_klasor = "/content/drive/MyDrive/Sesler"
db_klasoru = "/content/drive/MyDrive/Ses_Sifreleri_DB"

os.makedirs(db_klasoru, exist_ok=True)

print("\n2. Model Yükleniyor...")
device = "cuda" if torch.cuda.is_available() else "cpu"
kimlik_modeli = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb",
    run_opts={"device": device}
)

print("\n3. Sesler İşleniyor ve Şifreler Oluşturuluyor...")
ses_dosyalari = [d for d in os.listdir(kaynak_klasor) if d.endswith(('.mp3', '.wav'))]

if len(ses_dosyalari) == 0:
    print("HATA: Kaynak klasörde ses bulunamadı. Kısayolu doğru eklediğine emin ol!")
else:
    # Hızlıca test edebilmek için sadece ilk 5 dosyayı işliyoruz
    for dosya in ses_dosyalari[:5]:
        # Uzantıyı atıp dosya adını ID olarak alıyoruz
        kisi_adi = dosya.replace(".mp3", "").replace(".wav", "")
        dosya_yolu = os.path.join(kaynak_klasor, dosya)

        print(f"-> Şifre Kaydediliyor: {kisi_adi}")

        sinyal, ornekleme_hizi = torchaudio.load(dosya_yolu)
        if ornekleme_hizi != 16000:
            sinyal = torchaudio.transforms.Resample(orig_freq=ornekleme_hizi, new_freq=16000)(sinyal)

        with torch.no_grad():
            vektor = kimlik_modeli.encode_batch(sinyal.to(device)).squeeze()

        # .pt dosyasını veritabanına kaydet
        torch.save(vektor, f"{db_klasoru}/{kisi_adi}.pt")

    print("\n İŞLEM TAMAM! Şifreler başarıyla veritabanına kaydedildi.")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/hyperparams.yaml'


1. Klasörler Hazırlanıyor...

2. Model Yükleniyor...


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder



3. Sesler İşleniyor ve Şifreler Oluşturuluyor...
-> Şifre Kaydediliyor: common_voice_tr_43603498
-> Şifre Kaydediliyor: common_voice_tr_43603503
-> Şifre Kaydediliyor: common_voice_tr_43603507
-> Şifre Kaydediliyor: common_voice_tr_43603511
-> Şifre Kaydediliyor: common_voice_tr_43603512

🎉 İŞLEM TAMAM! Şifreler başarıyla veritabanına kaydedildi.


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/Ses_Sifreleri_DB"))

['common_voice_tr_43603498.pt', 'common_voice_tr_43603503.pt', 'common_voice_tr_43603507.pt', 'common_voice_tr_43603511.pt', 'common_voice_tr_43603512.pt']


In [ ]:
# Veritabanı (Şifrelerin olduğu) klasörün yolu
DB_KLASORU = "/content/drive/MyDrive/Ses_Sifreleri_DB"

def sisteme_giris_yap(kullanici_adi, okunan_ses_dosyasi, ekranda_yazan_sifre="Test Sifresi"):
    print(f"\n[{kullanici_adi}] için giriş talebi inceleniyor...")
    db_yolu = f"{DB_KLASORU}/{kullanici_adi}.pt"

    # ---------------- 1. KULLANICI KAYITLI MI? ----------------
    if not os.path.exists(db_yolu):
        return False, "HATA: Böyle bir kullanıcı sistemde kayıtlı değil!"

    # ---------------- 2. SES O KİŞİYE Mİ AİT? (KİMLİK DOĞRULAMA) ----------------
    sinyal, ornekleme_hizi = torchaudio.load(okunan_ses_dosyasi)
    if ornekleme_hizi != 16000:
        sinyal = torchaudio.transforms.Resample(orig_freq=ornekleme_hizi, new_freq=16000)(sinyal)

    with torch.no_grad():
        yeni_vektor = kimlik_modeli.encode_batch(sinyal.to(device)).squeeze()

    kayitli_vektor = torch.load(db_yolu, weights_only=True).to(device)

    benzerlik_skoru = F.cosine_similarity(yeni_vektor.unsqueeze(0), kayitli_vektor.unsqueeze(0)).item()

    if benzerlik_skoru < 0.25:
        return False, f"HATA: Ses eşleşmedi! (Skor: {benzerlik_skoru:.2f}) Yetkisiz giriş denemesi."

    # ---------------- 3. NE SÖYLEDİĞİNİ ANLAMA (TEST MODU) ----------------
    ses_matrisi_16k, _ = librosa.load(okunan_ses_dosyasi, sr=16000)
    girdiler = whisper_islemci(ses_matrisi_16k, sampling_rate=16000, return_tensors="pt")

    uretilen_idler = whisper_modeli.generate(girdiler.input_features, language="turkish", task="transcribe")
    soylenen_metin = whisper_islemci.batch_decode(uretilen_idler, skip_special_tokens=True)[0].strip().lower()

    # DİKKAT: Burada sadece bilgi yazdırıyoruz.
    print(f"   [BİLGİ] Duyulan Metin: '{soylenen_metin}'")
    print(f"   [TEST MODU] Metin eşleşmesine bakılmaksızın ses kimliği ile geçişe izin veriliyor...")

    # ---------------- 4. GİRİŞ BAŞARILI ----------------
    return True, f"BAŞARILI: Hoş geldin, {kullanici_adi}! (Ses Skoru: {benzerlik_skoru:.2f})"


# --- SİSTEMİ TEST ETME ZAMANI ---
# Mozilla'dan Drive'ına kaydettiğin bir kişinin ID'sini ve onun bir ses dosyasının yolunu buraya gir
test_kullanici = "common_voice_tr_43603498"
test_sesi = "/content/drive/MyDrive/Sesler/common_voice_tr_43603498.mp3"

giris_izni, mesaj = sisteme_giris_yap(test_kullanici, test_sesi)

print("\n" + "="*50)
print(mesaj)
print("="*50)


[common_voice_tr_43603498] için giriş talebi inceleniyor...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

   [BİLGİ] Duyulan Metin: 'güzel kokulut çeşeceklerin beyazdan gider.'
   [TEST MODU] Metin eşleşmesine bakılmaksızın ses kimliği ile geçişe izin veriliyor...

BAŞARILI: Hoş geldin, common_voice_tr_43603498! (Ses Skoru: 1.00)
